In [ ]:
import pandas as pd

# 1. 데이터 불러오기
weather_df = pd.read_csv('/content/merged_weather_pm_data_final_null_o.csv', parse_dates=['ymdt'])
traff_df = pd.read_csv('/content/지점별_시간대별_평균_지역구포함.csv')

# 2. traff_df를 long format으로 변환
traff_long = traff_df.melt(
    id_vars=['일자', '지역구', '지점번호'],
    value_vars=['3시', '8시', '14시', '18시'],
    var_name='시간',
    value_name='교통량'
)

# 3. 시간 정리: '3시' -> 3, '8시' -> 8 (int)
traff_long['시간'] = traff_long['시간'].str.replace('시', '').astype(int)

# 4. '일자' 문자열 -> datetime 변환
traff_long['일자'] = pd.to_datetime(traff_long['일자'].astype(str), format='%Y%m%d')

# 5. 시간 붙여서 datetime 컬럼 생성
traff_long['시간_dt'] = traff_long['시간'].apply(lambda h: pd.Timedelta(hours=h))
traff_long['datetime'] = traff_long['일자'] + traff_long['시간_dt']


# 6. 병합
merged_df = pd.merge(
    weather_df,
    traff_long[['datetime', '지역구', '교통량']],
    left_on=['ymdt', 'station_name'],
    right_on=['datetime', '지역구'],
    how='left'
)

# 7. 불필요한 컬럼 제거 및 확인
merged_df = merged_df.drop(columns=['datetime'])

print(merged_df.head(100))


     id station_name                ymdt  season park1_name  park1_dir_sin  \
0     1          강남구 2024-01-01 03:00:00  winter        서울숲  -3.162278e-01   
1     2          강동구 2024-01-01 03:00:00  winter      올림픽공원  -4.472136e-01   
2     3          강북구 2024-01-01 03:00:00  winter     북서울꿈의숲   6.507914e-01   
3     4          강서구 2024-01-01 03:00:00  winter      선유도공원   9.958932e-01   
4     5          관악구 2024-01-01 03:00:00  winter        현충원   3.713907e-01   
..  ...          ...                 ...     ...        ...            ...   
95   96          용산구 2024-01-01 18:00:00  winter       남산공원  -2.425356e-01   
96   97          은평구 2024-01-01 18:00:00  winter   안산도시자연공원   4.472136e-01   
97   98          종로구 2024-01-01 18:00:00  winter      푸른식물원   0.000000e+00   
98   99           중구 2024-01-01 18:00:00  winter       남산공원   8.320503e-01   
99  100          중랑구 2024-01-01 18:00:00  winter      중랑캠핑숲   1.224647e-16   

    park1_dir_cos  park1_distance  park1_area park2_name  ...  

In [ ]:
print(merged_df['지역구'].unique())
missing_gu = merged_df[merged_df['교통량'].isna()]['station_name'].unique()
print('병합 후 교통량 누락된 지역구:', missing_gu)
print(set(weather_df['station_name'].unique()) - set(traff_long['지역구'].unique()))



['강남구' '강동구' nan '강서구' '광진구' '구로구' '금천구' '노원구' '도봉구' '동대문구' '동작구' '마포구'
 '서초구' '성동구' '성북구' '송파구' '양천구' '영등포구' '용산구' '은평구' '종로구' '중구' '중랑구']
병합 후 교통량 누락된 지역구: ['강남구' '강북구' '강서구' '관악구' '동대문구' '서대문구' '성북구' '영등포구' '성동구' '도봉구' '종로구'
 '강동구' '서초구' '노원구' '마포구' '금천구' '중구' '용산구']
{'서대문구', '강북구', '관악구'}


In [ ]:
failed = merged_df[merged_df['교통량'].isna()]
print(failed.groupby('station_name').size())


station_name
강남구     770
강동구      46
강북구     832
강서구     109
관악구     832
금천구     108
노원구      20
도봉구      17
동대문구    275
마포구      44
서대문구    832
서초구       9
성동구      62
성북구     783
영등포구     41
용산구      28
종로구       9
중구        1
dtype: int64


In [ ]:
print(weather_df['ymdt'].dt.floor('H').head())
print(traff_long['datetime'].head())


0   2024-01-01 03:00:00
1   2024-01-01 03:00:00
2   2024-01-01 03:00:00
3   2024-01-01 03:00:00
4   2024-01-01 03:00:00
Name: ymdt, dtype: datetime64[ns]
0   2024-01-01 03:00:00
1   2024-01-01 03:00:00
2   2024-01-01 03:00:00
3   2024-01-01 03:00:00
4   2024-01-01 03:00:00
Name: datetime, dtype: datetime64[ns]


<ipython-input-10-28cf196fb7a7>:1: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  print(weather_df['ymdt'].dt.floor('H').head())


In [ ]:
total_rows = len(merged_df)
matched_rows = merged_df['교통량'].notna().sum()
print(f'병합 성공률: {matched_rows / total_rows:.2%}')


병합 성공률: 76.84%


In [ ]:
print(merged_df[merged_df['교통량'].isna()][['ymdt', 'station_name']].head(1000))


                    ymdt station_name
0    2024-01-01 03:00:00          강남구
2    2024-01-01 03:00:00          강북구
3    2024-01-01 03:00:00          강서구
4    2024-01-01 03:00:00          관악구
10   2024-01-01 03:00:00         동대문구
...                  ...          ...
4129 2024-03-13 08:00:00          관악구
4138 2024-03-13 08:00:00         서대문구
4141 2024-03-13 08:00:00          성북구
4150 2024-03-13 14:00:00          강남구
4152 2024-03-13 14:00:00          강북구

[1000 rows x 2 columns]


In [ ]:
merged_df.to_csv('/content/weather_traffic_merged_final_null_o.csv', index=False)